In [1]:
import pickle
from collections import defaultdict

with open("../data/AWESOME-PYTHON-REPOS_FINAL.pkl", "rb") as f:
    repos_dict = pickle.load(f)

cat_repos = defaultdict(list)
for repo_id, cats in repos_dict.items():
    for cat in cats:
        cat_repos[cat].append(repo_id)

print(f"Total repos: {len(repos_dict)}")
print(f"Total categories: {len(cat_repos)}")
print(f"\nCategory sizes (sorted):")
for cat, repos in sorted(cat_repos.items(), key=lambda x: -len(x[1])):
    print(f"  {len(repos):3d}  {cat}")

Total repos: 487
Total categories: 69

Category sizes (sorted):
   21  File Format Processing
   20  Science
   20  Testing
   16  AI and Agents
   16  Web Frameworks
   16  Data Visualization
   16  Code Analysis
   16  DevOps Tools
   16  Text Processing
   13  Database Drivers
   13  CLI Tools
   12  Data Analysis
   12  GUI Development
   11  Machine Learning
   11  ORM
   11  CLI Development
   10  Web APIs
   10  Debugging Tools
    9  Web Servers
    9  Web Scraping
    8  Image Processing
    8  Audio & Video Processing
    7  Deep Learning
    7  Authentication
    7  Admin Panels
    7  Asynchronous Programming
    6  Distributed Computing
    6  Algorithms and Design Patterns
    6  HTML Manipulation
    6  Implementations
    6  Functional Programming
    5  Geolocation
    5  HTTP Clients
    5  Database
    5  Data Validation
    5  Interactive Interpreter
    5  Build Tools
    5  Documentation
    5  Job Schedulers
    5  Package Management
    5  Distribution
    4  Co

In [3]:
import pickle
import random
from collections import defaultdict
from elasticsearch import Elasticsearch
import os
from dotenv import load_dotenv
import numpy as np

load_dotenv()

random.seed(42)  # Fixed seed for reproducibility

ES_URL = os.getenv("ES_URL", "http://localhost:9200")
API_KEY = os.getenv("ES_API_KEY")
INDEX = "repositories_enriched_new"

es = Elasticsearch(ES_URL, api_key=API_KEY)

with open("../data/AWESOME-PYTHON-REPOS_FINAL.pkl", "rb") as f:
    repos_dict = pickle.load(f)

# Build category -> [repo_id, ...] mapping
cat_repos = defaultdict(list)
for repo_id, cats in repos_dict.items():
    for cat in cats:
        cat_repos[cat].append(repo_id)


def is_valid(repo_id: str) -> bool:
    """Check if a repo has complete data in ES."""
    try:
        doc = es.get(index=INDEX, id=repo_id)["_source"]
    except Exception:
        return False

    # Check readme_summary
    if not doc.get("readme_summary"):
        return False

    # Check repo embedding (3072)
    emb = doc.get("embedding")
    if not emb or len(emb) != 3072 or np.allclose(emb, 0):
        return False

    # Check sub-embeddings (768)
    for field in ["embedding_code", "embedding_doc", "embedding_requirement", "embedding_readme"]:
        sub = doc.get(field)
        if not sub or len(sub) != 768 or np.allclose(sub, 0):
            return False

    return True


# Only select categories with at least MIN_CAT_SIZE repos
MIN_CAT_SIZE = 5
reference_repos = []

for cat, repos in sorted(cat_repos.items()):
    if len(repos) < MIN_CAT_SIZE:
        continue

    # Shuffle and pick first valid repo
    candidates = repos.copy()
    random.shuffle(candidates)
    selected = None
    for repo in candidates:
        if is_valid(repo):
            selected = repo
            break

    if selected:
        reference_repos.append(selected)
        print(f"  {cat} ({len(repos)}) -> {selected}")
    else:
        print(f"  [SKIP] {cat} ({len(repos)}) -> no valid repo found")

print(f"\nTotal reference repos: {len(reference_repos)}")
print(f"\nreference_repos = {reference_repos}")

  AI and Agents (16) -> pydantic/pydantic-ai
  Admin Panels (7) -> jet-admin/jet-bridge
  Algorithms and Design Patterns (6) -> grantjenks/python-sortedcontainers
  Asynchronous Programming (7) -> MagicStack/uvloop
  Audio & Video Processing (8) -> Zulko/moviepy
  Authentication (7) -> django-guardian/django-guardian
  Build Tools (5) -> pyinvoke/invoke
  CLI Development (11) -> Textualize/textual
  CLI Tools (13) -> httpie/cli
  Code Analysis (16) -> PyCQA/isort
  Data Analysis (12) -> dgunning/edgartools
  Data Validation (5) -> alecthomas/voluptuous
  Data Visualization (16) -> SciTools/cartopy
  Database (5) -> patx/pickledb
  Database Drivers (13) -> psycopg/psycopg
  Debugging Tools (10) -> inducer/pudb
  Deep Learning (7) -> Lightning-AI/pytorch-lightning
  DevOps Tools (16) -> getsentry/sentry-python
  Distributed Computing (6) -> mpi4py/mpi4py
  Distribution (5) -> dashingsoft/pyarmor
  Documentation (5) -> mkdocs/mkdocs
  File Format Processing (21) -> docling-project/docling

In [4]:
import pickle
import random
from collections import defaultdict

random.seed(42)

with open("../data/AWESOME-PYTHON-REPOS_FINAL.pkl", "rb") as f:
    repos_dict = pickle.load(f)

# Build category -> [repo_ids] mapping
cat_repos = defaultdict(list)
for repo_id, cats in repos_dict.items():
    for cat in cats:
        cat_repos[cat].append(repo_id)

REFERENCE_REPOS = [
    "pydantic/pydantic-ai", "jet-admin/jet-bridge", "grantjenks/python-sortedcontainers",
    "MagicStack/uvloop", "Zulko/moviepy", "django-guardian/django-guardian",
    "pyinvoke/invoke", "Textualize/textual", "httpie/cli", "PyCQA/isort",
    "dgunning/edgartools", "alecthomas/voluptuous", "SciTools/cartopy",
    "patx/pickledb", "psycopg/psycopg", "inducer/pudb", "Lightning-AI/pytorch-lightning",
    "getsentry/sentry-python", "mpi4py/mpi4py", "dashingsoft/pyarmor", "mkdocs/mkdocs",
    "docling-project/docling", "Suor/funcy", "r0x0r/pywebview", "geopy/geopy",
    "martinblech/xmltodict", "aio-libs/aiohttp", "libvips/pyvips", "python/cpython",
    "prompt-toolkit/python-prompt-toolkit", "sartography/SpiffWorkflow",
    "feature-engine/feature_engine", "BeanieODM/beanie", "pypa/pip",
    "networkx/networkx", "joke2k/faker", "sqids/sqids-python", "sanic-org/sanic",
    "Kludex/starlette", "scrapy/scrapy", "benoitc/gunicorn"
]

pairs = {}
for ref_repo in REFERENCE_REPOS:
    ref_cats = repos_dict.get(ref_repo, [])
    # Get all repos in same categories, exclude ref_repo itself
    candidates = set()
    for cat in ref_cats:
        candidates.update(cat_repos[cat])
    candidates.discard(ref_repo)
    candidates = list(candidates)
    partner = random.choice(candidates)
    pairs[ref_repo] = partner
    print(f"  {ref_repo} -> {partner}")

print(f"\nSEARCH_PAIRS = {pairs}")

  pydantic/pydantic-ai -> TauricResearch/TradingAgents
  jet-admin/jet-bridge -> offerrall/FuncToWeb
  grantjenks/python-sortedcontainers -> pytransitions/transitions
  MagicStack/uvloop -> agronholm/anyio
  Zulko/moviepy -> sergree/matchering
  django-guardian/django-guardian -> authlib/authlib
  pyinvoke/invoke -> pydoit/doit
  Textualize/textual -> fastapi/typer
  httpie/cli -> tmux/tmux
  PyCQA/isort -> astral-sh/ty
  dgunning/edgartools -> pathwaycom/pathway
  alecthomas/voluptuous -> pyeve/cerberus
  SciTools/cartopy -> matplotlib/matplotlib
  patx/pickledb -> zopefoundation/ZODB
  psycopg/psycopg -> microsoft/mssql-python
  inducer/pudb -> ionelmc/python-manhole
  Lightning-AI/pytorch-lightning -> ChristosChristofidis/awesome-deep-learning
  getsentry/sentry-python -> boto/boto3
  mpi4py/mpi4py -> dask/dask
  dashingsoft/pyarmor -> pyinstaller/pyinstaller
  mkdocs/mkdocs -> mitmproxy/pdoc
  docling-project/docling -> yfedoseev/pdf_oxide
  Suor/funcy -> erikrose/more-itertools
  